In [ ]:
pip install "sunpy[net]"

In [2]:
import numpy as np
import matplotlib.pylab as plt
import matplotlib.mlab as mlab
import pandas as pd
import scipy.stats
import requests
import urllib
import json
from datetime import datetime as dt_obj
from datetime import timedelta
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from sunpy.time import TimeRange
from sunpy.net import Fido, attrs as a
import json
# 加入特征选择
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif

The following packages are not installed:
['drms>=0.7.1', 'zeep>=4.3.0']
To install sunpy with these dependencies use `pip install sunpy[net]` or `pip install sunpy[all]` for all extras. 
If you installed sunpy via conda, please report this to the community channel: https://matrix.to/#/#sunpy:openastronomy.org [sunpy.util.sysinfo]


ModuleNotFoundError: No module named 'zeep'

In [16]:
# Load data written by A2_read
CME_data = np.loadtxt('/coursedata/A2/CMEs_24_BI.csv', delimiter=',')
no_CME_data = np.loadtxt('/coursedata/A2/noCMEs_24_BI.csv', delimiter=',')

Normalization using median and std dev of features
--------------------------------------------------

In [17]:
def normalize_the_data(flare_data):
    flare_data = np.array(flare_data)
    n_elements = flare_data.shape[0]
    for j in range(flare_data.shape[1]):
        standard_deviation_of_this_feature = np.std(flare_data[:, j])
        median_of_this_feature = np.median(flare_data[:, j])
        for i in range(n_elements):
            flare_data[i, j] = (
                flare_data[i, j] - median_of_this_feature) / (3.0*standard_deviation_of_this_feature)
    return flare_data

In [21]:
# #All data normalized in one bunch
# aCME=np.concatenate((CME_data,no_CME_data))
# naCME = normalize_the_data(aCME)
# CME_data = naCME[0:CME_data.shape[0],:]
# no_CME_data = naCME[CME_data.shape[0]+1:,:]



print("There are", no_CME_data.shape[0], "flares that do not erupt.")
print("There are", CME_data.shape[0], "flares that erupt.")

There are 308 flares that do not erupt.
There are 51 flares that erupt.


Build X and y
-------------

In [23]:
N_features = 18
Nfl = CME_data.shape[0]
Nnofl = no_CME_data.shape[0]
yfl = np.ones(Nfl)
ynofl = np.zeros(Nnofl)
# X = np.vstack((CME_data, no_CME_data))
# y = np.hstack((
#     np.ones(CME_data.shape[0]),      # positive class = 1
#     np.zeros(no_CME_data.shape[0])   # negative class = 0
# ))

# print("Total X shape:", X.shape)
# print("Total y shape:", y.shape)
# print("Positive samples:", np.sum(y == 1))
# print("Negative samples:", np.sum(y == 0))

SVM model
---------

In [ ]:
# Import MyPreprocessor class from preprocessor_class.py
from preprocessor_class import MyPreprocessor

# Define feature columns (SHARP features)
FEATURES = ['USFLUX', 'MEANGBT', 'MEANJZH', 'MEANPOT', 'SHRGT45', 'TOTUSJH',
            'MEANGBH', 'MEANALP', 'MEANGAM', 'MEANGBZ', 'MEANJZD', 'TOTUSJZ',
            'SAVNCPP', 'TOTPOT', 'MEANSHR', 'AREA_ACR', 'R_VALUE', 'ABSNJZH']

# Create and build model using MyPreprocessor with feature columns
preprocessor = MyPreprocessor(feature_columns=FEATURES, use_svm=True, k_best=12)
clf = preprocessor.build_model()

Function to compute the confusion table
---------------------------------------

In [29]:
def confusion_table(pred, labels):
    Nobs = len(pred)
    TN = 0.
    TP = 0.
    FP = 0.
    FN = 0.
    for i in range(Nobs):
        if (pred[i] == 0 and labels[i] == 0):
            TN += 1
        elif (pred[i] == 1 and labels[i] == 0):
            FP += 1
        elif (pred[i] == 1 and labels[i] == 1):
            TP += 1
        elif (pred[i] == 0 and labels[i] == 1):
            FN += 1
        else:
            print("ERROR")
    return TN, FP, TP, FN

In [30]:
# lists to hold the TSS and standard deviation of the TSS
array_of_avg_TSS = np.ndarray([50])
array_of_std_TSS = np.ndarray([50])

# xdata are the features
# ydata are the labels
xdata = np.concatenate((CME_data, no_CME_data), axis=0)
ydata = np.concatenate((np.ones(Nfl), np.zeros(Nnofl)), axis=0)

# compute the TSS for a variety of fold numbers ranging from 2 to 51
for nfold in range(2, 52):
    skf = StratifiedKFold(n_splits=nfold, shuffle=True)
    these_TSS_for_this_nfold = []

    for train_index, test_index in skf.split(xdata, ydata):
        # xtrain are the features in the training set
        xtrain = xdata[train_index]
        # ytrain are the labels in the training set
        ytrain = ydata[train_index]

        # xtest are the features in the testing set
        xtest = xdata[test_index]
        ytest = ydata[test_index]

        clf.fit(xtrain, ytrain)
        pred = clf.predict(xtest)

        TN, FP, TP, FN = confusion_table(pred, ytest)

        if (((TP + FN) == 0.0) or ((FP + TN) == 0.0)):
            these_TSS_for_this_nfold.append(np.nan)
        else:
            these_TSS_for_this_nfold.append(TP/(TP+FN) - FP/(FP+TN))

    TSS_k = np.array(these_TSS_for_this_nfold)
    array_of_avg_TSS[nfold - 2] = np.mean(TSS_k)
    array_of_std_TSS[nfold - 2] = np.std(TSS_k)

After having investigated TSS(k), one can conclude that around k=10 the error bars are still small while TSS has attained a roughly constant value. Choosing that one as TSS and the corresponding std dev of TSS as "accuracy".

In [32]:
target_nfold = 11
print("The TSS equals", array_of_avg_TSS[target_nfold - 2],
      "plus or minus", array_of_std_TSS[target_nfold - 2], ".")

The TSS equals 0.251948051948052 plus or minus 0.21476334019595492 .


Grading section (TBA)
=====================

In [ ]:
# Save the model for submission
import joblib

# Define feature columns (SHARP features)
FEATURES = ['USFLUX', 'MEANGBT', 'MEANJZH', 'MEANPOT', 'SHRGT45', 'TOTUSJH',
            'MEANGBH', 'MEANALP', 'MEANGAM', 'MEANGBZ', 'MEANJZD', 'TOTUSJZ',
            'SAVNCPP', 'TOTPOT', 'MEANSHR', 'AREA_ACR', 'R_VALUE', 'ABSNJZH']
TARGET = "CME"

# Fit on full dataset
clf.fit(xdata, ydata)

# Create bundle following the reference format
bundle = {
    "target": TARGET,
    "model": clf,
    "preprocessor": MyPreprocessor(FEATURES),
}

joblib.dump(bundle, "student_model.joblib")
print("Saved student_model.joblib")

Save the model for submission
=========================